#Call Library

In [6]:
#Import needed libraries
import pandas as pd
import numpy as np
import os


In [7]:
# Create Tables directory if it doesn't exist
os.makedirs("Tables/Gold", exist_ok=True)

print("Tables directory created successfully")


Tables directory created successfully


#Call Data

In [8]:
Silver_path = r"/content/Cleaned Data.xlsx"

# Read Excel file using pandas
df = pd.read_excel(Silver_path)

print(f"Data loaded successfully. Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")


Data loaded successfully. Shape: (5, 3)
Columns: ['Account Title', 'Market Place', 'Market_ID']


In [9]:
#Check all sheets in the Excel file
excel_file_path = r"/content/Cleaned Data.xlsx"
xls = pd.ExcelFile(excel_file_path)
print("Available sheets:")
print(xls.sheet_names)

# Display data from each sheet to identify date information
for sheet in xls.sheet_names:
    print(f"\n--- Sheet: {sheet} ---")
    sheet_df = pd.read_excel(excel_file_path, sheet_name=sheet)
    print(f"Shape: {sheet_df.shape}")
    print(f"Columns: {sheet_df.columns.tolist()}")
    print(sheet_df.head(3))


Available sheets:
['Dim_market_account', 'Dim_product', 'Dim_date', 'Fact', 'Sales']

--- Sheet: Dim_market_account ---
Shape: (5, 3)
Columns: ['Account Title', 'Market Place', 'Market_ID']
  Account Title Market Place  Market_ID
0   [ UK ] Aava           UK          0
1   [ FR ] Aava           FR          1
2   [ DE ] Aava           DE          2

--- Sheet: Dim_product ---
Shape: (73, 8)
Columns: ['Title', 'SKU', 'SKU_ID', 'FNSKU', 'ASIN', 'Parent ASIN', 'Is Parent', 'Brand']
                                               Title  \
0  High Absorption Magnesium Citrate Supplements ...   
1  Omega 3 2000mg Fish Oil Capsules - High Streng...   
2  Turmeric Curcumin Capsules with Black Pepper [...   

                              SKU  SKU_ID       FNSKU        ASIN Parent ASIN  \
0  UK_Aava_Magnesium_400mg_120_05       0  X000JJIQ79  B01GU9QLZC   No Parent   
1     UK_Aava_Omega_2000mg_120_05       1  X000JFSY57  B01GU9P0M2   No Parent   
2  UK_Aava_Turmeric_1400mg_120_07       2  X000J6

#Create Dim

In [10]:
# Create Dimension Tables

# Create Market Dimension
dim_Market = df[["Market_ID", "Market Place"]].drop_duplicates().reset_index(drop=True)
dim_Market.insert(0, "dim_Market_id", range(1, len(dim_Market) + 1))

# Create Account Dimension
dim_Account = df[["Account Title"]].drop_duplicates().reset_index(drop=True)
dim_Account.insert(0, "dim_Account_id", range(1, len(dim_Account) + 1))

# Save dimensions
dim_Market.to_csv("Tables/Gold/dim_Market.csv", index=False)
dim_Market.to_parquet("Tables/Gold/dim_Market.parquet", index=False)

dim_Account.to_csv("Tables/Gold/dim_Account.csv", index=False)
dim_Account.to_parquet("Tables/Gold/dim_Account.parquet", index=False)

print("\n=== DIMENSION TABLES ===")
print("\n--- dim_Market ---")
print(dim_Market)

print("\n--- dim_Account ---")
print(dim_Account)



=== DIMENSION TABLES ===

--- dim_Market ---
   dim_Market_id  Market_ID Market Place
0              1          0           UK
1              2          1           FR
2              3          2           DE
3              4          3           IT
4              5          4           ES

--- dim_Account ---
   dim_Account_id Account Title
0               1   [ UK ] Aava
1               2   [ FR ] Aava
2               3   [ DE ] Aava
3               4   [ IT ] Aava
4               5   [ ES ] Aava


In [11]:
# Create Date Dimension from original data

# Read Dim_date from the original Star Schema file
dim_Date = pd.read_excel(Silver_path, sheet_name='Dim_date')

# Reorder columns to match expected structure
dim_Date = dim_Date[['Date_ID', 'Date', 'Year', 'Month', 'Quarter', 'Day']]

# Save dimension
dim_Date.to_csv("Tables/Gold/dim_Date.csv", index=False)
dim_Date.to_parquet("Tables/Gold/dim_Date.parquet", index=False)

print("\n=== DATE DIMENSION (from Original Data: 2020-2021) ===")
print(f"\nTotal records: {len(dim_Date)}")
print(f"Date range: {dim_Date['Date'].min()} to {dim_Date['Date'].max()}")
print(f"\nFirst 10 records:")
print(dim_Date.head(10))
print(f"\nLast 10 records:")
print(dim_Date.tail(10))



=== DATE DIMENSION (from Original Data: 2020-2021) ===

Total records: 209
Date range: 2020-10-01 00:00:00 to 2021-04-27 00:00:00

First 10 records:
    Date_ID       Date  Year  Month  Quarter  Day
0  10012020 2020-10-01  2020     10        4    1
1  10022020 2020-10-02  2020     10        4    2
2  10032020 2020-10-03  2020     10        4    3
3  10042020 2020-10-04  2020     10        4    4
4  10052020 2020-10-05  2020     10        4    5
5  10062020 2020-10-06  2020     10        4    6
6  10072020 2020-10-07  2020     10        4    7
7  10082020 2020-10-08  2020     10        4    8
8  10092020 2020-10-09  2020     10        4    9
9  10102020 2020-10-10  2020     10        4   10

Last 10 records:
     Date_ID       Date  Year  Month  Quarter  Day
199  4182021 2021-04-18  2021      4        2   18
200  4192021 2021-04-19  2021      4        2   19
201  4202021 2021-04-20  2021      4        2   20
202  4212021 2021-04-21  2021      4        2   21
203  4222021 2021-04-22  20

In [12]:
# Create Fact Table

# Create a mapping dataframe from original data
fact_AccountMarket = df.copy()

# Add surrogate keys from dimensions
fact_AccountMarket = fact_AccountMarket.merge(
    dim_Account[["dim_Account_id", "Account Title"]],
    on="Account Title",
    how="left"
)

fact_AccountMarket = fact_AccountMarket.merge(
    dim_Market[["dim_Market_id", "Market_ID"]],
    on="Market_ID",
    how="left"
)

# Create fact table with foreign keys
fact_table = fact_AccountMarket[["dim_Account_id", "dim_Market_id"]].copy()
fact_table.insert(0, "fact_AccountMarket_id", range(1, len(fact_table) + 1))

# Save fact table
fact_table.to_csv("Tables/Gold/fact_AccountMarket.csv", index=False)
fact_table.to_parquet("Tables/Gold/fact_AccountMarket.parquet", index=False)

print("\n=== FACT TABLE ===")
print("\n--- fact_AccountMarket ---")
print(fact_table)

print("\n\nSchema Summary:")
print(f"Fact Table Records: {len(fact_table)}")
print(f"Unique Accounts (dim_Account): {dim_Account['dim_Account_id'].max()}")
print(f"Unique Markets (dim_Market): {dim_Market['dim_Market_id'].max()}")



=== FACT TABLE ===

--- fact_AccountMarket ---
   fact_AccountMarket_id  dim_Account_id  dim_Market_id
0                      1               1              1
1                      2               2              2
2                      3               3              3
3                      4               4              4
4                      5               5              5


Schema Summary:
Fact Table Records: 5
Unique Accounts (dim_Account): 5
Unique Markets (dim_Market): 5


In [13]:
df

,Account Title,Market Place,Market_ID
0,[ UK ] Aava,UK,0
1,[ FR ] Aava,FR,1
2,[ DE ] Aava,DE,2
3,[ IT ] Aava,IT,3
4,[ ES ] Aava,ES,4


In [15]:
# Export Star Schema to Excel File

import os
import gc
import time

excel_file = "Tables/Gold/Star_Schema_Updated.xlsx"

# Force garbage collection to release file handles
gc.collect()
time.sleep(0.5)

# Create new Excel file with updated data
with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    # Write each table to a separate sheet
    dim_Date.to_excel(writer, sheet_name='dim_Date', index=False)
    dim_Account.to_excel(writer, sheet_name='dim_Account', index=False)
    dim_Market.to_excel(writer, sheet_name='dim_Market', index=False)
    fact_AccountMarket.to_excel(writer, sheet_name='fact_AccountMarket', index=False)

print(f"✓ Star Schema exported successfully to: {excel_file}")
print(f"\nSheets created:")
print(f"  - dim_Date ({len(dim_Date)} rows) - Data: 2020-10-01 to 2021-04-27")
print(f"  - dim_Account ({len(dim_Account)} rows)")
print(f"  - dim_Market ({len(dim_Market)} rows)")
print(f"  - fact_AccountMarket ({len(fact_AccountMarket)} rows)")


✓ Star Schema exported successfully to: Tables/Gold/Star_Schema_Updated.xlsx

Sheets created:
  - dim_Date (209 rows) - Data: 2020-10-01 to 2021-04-27
  - dim_Account (5 rows)
  - dim_Market (5 rows)
  - fact_AccountMarket (5 rows)
